# SNODAS SWE Data Processing and Integration for HUC12s

This notebook downloads daily SNODAS SWE (Snow Water Equivalent) data, aggregates it to HUC12 watershed boundaries, and writes the results as monthly and combined parquet files directly to S3. It then merges the SNODAS SWE values into model-ready HUC12 CSVs for furthermodeling. This notebook includes downloading and parsing SNODAS data, spatial aggregation using rasterization, efficient S3 storage, and updating model-ready datasets with the new SWE information.

In [1]:
import os
import re
import io
import gzip
import tarfile
import math
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import boto3

from bs4 import BeautifulSoup
from pathlib import Path
from datetime import date, datetime
from rasterio.transform import from_bounds
from rasterio.features import rasterize
from rasterio.windows import from_bounds as window_from_bounds
from snowML.datapipe.utils import data_utils as du
from snowML.datapipe.utils import get_geos as gg
from affine import Affine
import s3fs

/home/sagemaker-user/.conda/envs/myenv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
# --- Config: SNODAS (G02158) download, HUC12 aggregation, S3 upload ---
# Data: https://noaadata.apps.nsidc.org/NOAA/G02158/

BUCKET_NAME = "snowml-shape"
BRONZE_BUCKET = "snowml-bronze"
MODEL_READY_BUCKET = "snowml-model-ready-stgnn"
AWS_REGION = "us-west-2"

# --- Region / HUCs ---
HUC_LEVEL = "12"
HUC8_IDS = [
    17020009, 17020011, 17110005, 17110006, 17110009,
    17010304, 17010302, 17060207, 17060208, 17030001, 17030002,
    17110008, 17030003, 17020010, 17110007,
]
EXCLUDED_HUCS_CA = [
    "171100050101", "171100050102", "171100050201", "171100050202", "171100050203",
    "171100050301", "171100050302", "171100050303", "171100050304", "171100050305",
    "171100050306", "171100050401",
]

WEIRD_HUCS = []
EXCLUDED_HUCS = EXCLUDED_HUCS_CA + WEIRD_HUCS

In [3]:
dhsvm_lidar_hucs = pd.read_csv("dhsvm_lidar_hucs.csv")
hucids_not_in_training = dhsvm_lidar_hucs.loc[dhsvm_lidar_hucs["not_in_training"] == 1, "huc_id"].tolist()
len(hucids_not_in_training)

91

In [4]:
def get_geo_jsons(huc_ids, huc_level, bucket_nm="snowml-shape"):
    gdf = []
    for huc_id in huc_ids:
        temp = None
        f_out = f"Huc{huc_level}_in_{huc_id}.geojson"
        if du.isin_s3(bucket_nm, f_out):
            print(f"Getting geos from s3 bucket for {huc_id}")
            temp = du.s3_to_gdf(bucket_nm, f_out)
        else:
            try:
                print(f"Getting geos from google earth for {huc_id}")
                temp = gg.get_geos(huc_id, huc_level, s3_save=False, bucket_nm=bucket_nm)
            except Exception as e:
                # log the error and continue; store None for this huc_id
                print(f"Error getting geos for {huc_id}: {e}")
        if temp is not None:
            temp['huc8'] = huc_id  # Add huc8 df column for all rows in this GeoDataFrame
            gdf.append(temp)
            print(f"Found {len(temp)} HUC {huc_level}s in {huc_id}")
    return gpd.GeoDataFrame(pd.concat(gdf, ignore_index=True))

In [5]:
# 1. Load HUC12 geometries from snowml-shape
gdf = get_geo_jsons([170103020101], HUC_LEVEL, BUCKET_NAME)
gdf = gdf[~gdf["huc_id"].isin(EXCLUDED_HUCS)]
print(f"Loaded {len(gdf)} HUC12 basins")

Getting geos from google earth for 170103020101
Found 1 HUC 12s in 170103020101
Loaded 1 HUC12 basins


In [8]:
# =========================
# CONFIG
# =========================
BASE_URL = "https://noaadata.apps.nsidc.org/NOAA/G02158/masked"
OUT_RAW_DIR = Path("snodas_raw")
OUT_HUC_DIR = Path("snodas_huc12_csvs")
OUT_RAW_DIR.mkdir(exist_ok=True, parents=True)
OUT_HUC_DIR.mkdir(exist_ok=True, parents=True)

DATE_START = "2003-09-29"
DATE_END = "2026-03-10"   # full range 2003–2026; only available days will download

# S3: parquet written directly to S3 (no local parquet files)
SNODAS_PARQUET_S3_BUCKET = BRONZE_BUCKET
SNODAS_S3_PREFIX = "snodas"
SNODAS_PARQUET_S3_KEY = f"{SNODAS_S3_PREFIX}/HUC12_SNODAS_SWE_2003_2026.parquet"

# Keep huc_id as string
gdf["huc_id"] = gdf["huc_id"].astype(str)
gdf = gdf.to_crs("EPSG:4326").copy()

# =========================
# SNODAS CONSTANTS
# =========================
SNODAS_ROWS = 3351
SNODAS_COLS = 6935
SNODAS_NODATA = -9999

# Masked grid bounds from NSIDC user guide
# Before 2013-10-01:
PRE_SHIFT_BOUNDS = {
    "west": -124.7337,
    "south": 24.9504,
    "east": -66.9421,
    "north": 52.8754,
}
# 2013-10-01 and later:
POST_SHIFT_BOUNDS = {
    "west": -124.7333,
    "south": 24.9500,
    "east": -66.9417,
    "north": 52.8750,
}

SHIFT_DATE = pd.Timestamp("2013-10-01")

# Known errata / missing dates from NSIDC text file
KNOWN_BAD_DATES = {
    "2004-02-25", "2004-08-31", "2004-09-27", "2005-06-25", "2005-08-01", "2005-08-02",
    "2006-08-26", "2006-08-27", "2006-09-08", "2006-09-30", "2006-10-01", "2007-02-14",
    "2007-03-26", "2008-03-13", "2008-06-13", "2008-06-18", "2009-08-20", "2012-12-20",
    # bad individual SWE file / bad erratum
    "2003-10-30",
    "2005-07-31",
}


In [7]:

# =========================
# HELPERS
# =========================
def month_folder(dt: pd.Timestamp) -> str:
    return dt.strftime("%m_%b")

def get_masked_bounds(dt: pd.Timestamp) -> dict:
    return PRE_SHIFT_BOUNDS if dt < SHIFT_DATE else POST_SHIFT_BOUNDS

def get_transform(dt: pd.Timestamp) -> Affine:
    b = get_masked_bounds(dt)
    return from_bounds(
        b["west"], b["south"], b["east"], b["north"],
        SNODAS_COLS, SNODAS_ROWS
    )

def daterange(start_date: str, end_date: str):
    dates = pd.date_range(start=start_date, end=end_date, freq="D")
    for d in dates:
        yield pd.Timestamp(d)

def list_month_tar_urls(session: requests.Session, year: int, month: int) -> dict:
    """
    Returns dict: { 'YYYY-MM-DD': file_url }
    """
    dt = pd.Timestamp(year=year, month=month, day=1)
    url = f"{BASE_URL}/{year}/{month_folder(dt)}/"
    r = session.get(url, timeout=60)
    if r.status_code != 200:
        return {}

    soup = BeautifulSoup(r.text, "html.parser")
    out = {}
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.startswith("SNODAS_") and href.endswith(".tar"):
            m = re.search(r"(\d{8})", href)
            if m:
                dstr = datetime.strptime(m.group(1), "%Y%m%d").strftime("%Y-%m-%d")
                out[dstr] = url + href
    return out

def download_file(session: requests.Session, url: str, local_path: Path):
    local_path.parent.mkdir(parents=True, exist_ok=True)
    if local_path.exists():
        return local_path
    with session.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(local_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    return local_path

def download_tar_bytes(session: requests.Session, url: str) -> bytes:
    """
    Download one tar file to memory instead of saving to disk.
    """
    with session.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        chunks = []
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                chunks.append(chunk)
        return b"".join(chunks)

def read_swe_mm_from_tar(tar_path: Path) -> np.ndarray | None:
    """
    Returns SWE array in mm as float32, nodata stays as -9999.
    Uses correct SWE product code: 1034.
    """
    with tarfile.open(tar_path, "r:*") as tf:
        names = tf.getnames()

        # Correct SNODAS SWE member
        candidates = [n for n in names if "us_ssmv11034" in n and n.endswith(".dat.gz")]
        if not candidates:
            # fallback in case archive contains .dat directly
            candidates = [n for n in names if "us_ssmv11034" in n and n.endswith(".dat")]

        if not candidates:
            return None

        member = candidates[0]
        fobj = tf.extractfile(member)
        if fobj is None:
            return None

        raw = fobj.read()
        if member.endswith(".gz"):
            raw = gzip.decompress(raw)

        arr = np.frombuffer(raw, dtype=">i2")
        if arr.size != SNODAS_ROWS * SNODAS_COLS:
            raise ValueError(f"Unexpected SNODAS array size in {tar_path.name}: {arr.size}")

        arr = arr.reshape(SNODAS_ROWS, SNODAS_COLS).astype(np.float32)
        return arr

def read_swe_mm_from_tar_bytes(tar_bytes: bytes) -> np.ndarray | None:
    """
    Extract SWE array from in-memory tar bytes.
    Returns SWE in mm as float32.
    Nodata remains -9999.
    """
    with tarfile.open(fileobj=io.BytesIO(tar_bytes), mode="r:*") as tf:
        names = tf.getnames()

        candidates = [n for n in names if "us_ssmv11034" in n and n.endswith(".dat.gz")]
        if not candidates:
            candidates = [n for n in names if "us_ssmv11034" in n and n.endswith(".dat")]

        if not candidates:
            return None

        member = candidates[0]
        fobj = tf.extractfile(member)
        if fobj is None:
            return None

        raw = fobj.read()
        if member.endswith(".gz"):
            raw = gzip.decompress(raw)

        arr = np.frombuffer(raw, dtype=">i2")
        expected_size = SNODAS_ROWS * SNODAS_COLS
        if arr.size != expected_size:
            raise ValueError(f"Unexpected SNODAS array size: got {arr.size}, expected {expected_size}")

        arr = arr.reshape(SNODAS_ROWS, SNODAS_COLS).astype(np.float32)
        return arr

def build_zone_setup(gdf4326: gpd.GeoDataFrame, dt: pd.Timestamp):
    """
    Precompute clipped window + rasterized zone grid for a given SNODAS grid geometry.
    """
    transform = get_transform(dt)

    minx, miny, maxx, maxy = gdf4326.total_bounds

    pad_x = abs(transform.a) * 2
    pad_y = abs(transform.e) * 2

    win = window_from_bounds(
        minx - pad_x, miny - pad_y, maxx + pad_x, maxy + pad_y,
        transform=transform
    ).round_offsets().round_lengths()

    row_off = max(0, int(win.row_off))
    col_off = max(0, int(win.col_off))
    height = min(SNODAS_ROWS - row_off, int(win.height))
    width = min(SNODAS_COLS - col_off, int(win.width))

    clipped_transform = transform * Affine.translation(col_off, row_off)

    huc_ids = gdf4326["huc_id"].tolist()
    zone_lookup = {huc_id: i + 1 for i, huc_id in enumerate(huc_ids)}
    inv_lookup = {v: k for k, v in zone_lookup.items()}

    shapes = (
        (geom, zone_lookup[huc_id])
        for geom, huc_id in zip(gdf4326.geometry, gdf4326["huc_id"])
    )

    zone_raster = rasterize(
        shapes=shapes,
        out_shape=(height, width),
        transform=clipped_transform,
        fill=0,
        dtype="int32",
        all_touched=False
    )

    row_centers = np.array([
        clipped_transform.f + (r + 0.5) * clipped_transform.e
        for r in range(height)
    ])
    lat_weights = np.cos(np.deg2rad(row_centers)).astype(np.float32)
    weight_2d = np.repeat(lat_weights[:, None], width, axis=1)

    return {
        "row_off": row_off,
        "col_off": col_off,
        "height": height,
        "width": width,
        "zone_raster": zone_raster,
        "zone_lookup": zone_lookup,
        "inv_lookup": inv_lookup,
        "weight_2d": weight_2d,
    }

def aggregate_day_to_huc12(arr_mm: np.ndarray, zone_setup: dict) -> pd.DataFrame:
    """
    Returns DataFrame with columns: huc_id, swe_m
    Aggregation = approximate area-weighted mean SWE.
    """
    r0 = zone_setup["row_off"]
    c0 = zone_setup["col_off"]
    h = zone_setup["height"]
    w = zone_setup["width"]

    sub = arr_mm[r0:r0+h, c0:c0+w]
    zones = zone_setup["zone_raster"]
    weights = zone_setup["weight_2d"]

    valid = (zones > 0) & (sub != SNODAS_NODATA)
    if not np.any(valid):
        return pd.DataFrame(columns=["huc_id", "swe_m"])

    z = zones[valid]
    swe_mm = sub[valid]
    wt = weights[valid]

    weighted_sum = np.bincount(z, weights=swe_mm * wt)
    weight_sum = np.bincount(z, weights=wt)

    n_zones = max(zone_setup["inv_lookup"].keys())
    if len(weighted_sum) < n_zones + 1:
        weighted_sum = np.pad(weighted_sum, (0, n_zones + 1 - len(weighted_sum)))
    if len(weight_sum) < n_zones + 1:
        weight_sum = np.pad(weight_sum, (0, n_zones + 1 - len(weight_sum)))

    rows = []
    for zid in range(1, n_zones + 1):
        huc_id = zone_setup["inv_lookup"][zid]
        if weight_sum[zid] > 0:
            swe_m = (weighted_sum[zid] / weight_sum[zid]) / 1000.0
        else:
            swe_m = np.nan
        rows.append({"huc_id": huc_id, "swe_m": swe_m})

    return pd.DataFrame(rows)

def _month_s3_key(prefix: str, year: int, month: int) -> str:
    return f"{prefix}/monthly/snodas_huc12_swe_{year}_{month:02d}.parquet"

def append_month_chunk_to_s3(
    rows: list[pd.DataFrame],
    bucket: str,
    prefix: str,
    year: int,
    month: int,
    fs: s3fs.S3FileSystem,
) -> None:
    """Flush in-memory daily dataframes to a monthly parquet on S3. Merges with existing if present."""
    if not rows:
        return
    df = pd.concat(rows, ignore_index=True)[["huc_id", "date", "swe_m"]]
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["date", "huc_id"]).reset_index(drop=True)
    key = _month_s3_key(prefix, year, month)
    path = f"s3://{bucket}/{key}"
    if fs.exists(path):
        old = pd.read_parquet(path, storage_options={"client_kwargs": {"region_name": "us-west-2"}})
        df = pd.concat([old, df], ignore_index=True)
        df = df.drop_duplicates(subset=["huc_id", "date"], keep="last")
        df = df.sort_values(["date", "huc_id"]).reset_index(drop=True)
    with fs.open(path, "wb") as f:
        df.to_parquet(f, index=False)
    print(f"Wrote month {year}-{month:02d} -> s3://{bucket}/{key}")

def combine_monthly_parquets_from_s3(
    bucket: str,
    prefix: str,
    combined_key: str,
    fs: s3fs.S3FileSystem,
) -> pd.DataFrame:
    """Read all monthly parquets from S3, combine, dedupe, sort; write combined parquet to S3."""
    monthly_prefix = f"{bucket}/{prefix}/monthly/"
    listed = fs.ls(monthly_prefix)
    keys = sorted([k for k in listed if k.endswith(".parquet")])
    if not keys:
        raise RuntimeError(f"No monthly parquet files found under s3://{monthly_prefix}")
    dfs = [pd.read_parquet(f"s3://{k}", storage_options={"client_kwargs": {"region_name": "us-west-2"}}) for k in keys]
    df_all = pd.concat(dfs, ignore_index=True)
    df_all["date"] = pd.to_datetime(df_all["date"])
    df_all = df_all.drop_duplicates(subset=["huc_id", "date"], keep="last")
    df_all = df_all.sort_values(["date", "huc_id"]).reset_index(drop=True)
    combined_path = f"s3://{bucket}/{combined_key}"
    with fs.open(combined_path, "wb") as f:
        df_all.to_parquet(f, index=False)
    print(f"Wrote combined -> s3://{bucket}/{combined_key}")
    return df_all

def write_one_file_per_huc12(df_all: pd.DataFrame, out_dir: Path):
    out_dir.mkdir(exist_ok=True, parents=True)
    for huc_id, sub in df_all.groupby("huc_id"):
        sub = sub.sort_values("date").reset_index(drop=True)
        out_path = out_dir / f"{huc_id}_snodas_swe_2003_2026.csv"
        sub.to_csv(out_path, index=False)



In [ ]:
def run_snodas_huc12_pipeline(
    gdf4326: gpd.GeoDataFrame,
    start_date: str = DATE_START,
    end_date: str = DATE_END,
    bucket: str = SNODAS_PARQUET_S3_BUCKET,
    prefix: str = SNODAS_S3_PREFIX,
    combined_key: str = SNODAS_PARQUET_S3_KEY,
    combine_at_end: bool = True,
) -> pd.DataFrame | None:
    """
    Pipeline: download SNODAS by day (in memory), aggregate to HUC12, write parquet directly to S3.
    - Monthly shards -> s3://bucket/prefix/monthly/snodas_huc12_swe_YYYY_MM.parquet
    - If combine_at_end: reads monthly from S3, writes combined -> s3://bucket/combined_key
    """
    session = requests.Session()
    fs = s3fs.S3FileSystem(client_kwargs={"region_name": AWS_REGION})
    month_cache = {}

    zone_setup_pre = build_zone_setup(gdf4326, pd.Timestamp("2013-09-30"))
    zone_setup_post = build_zone_setup(gdf4326, pd.Timestamp("2013-10-01"))

    current_month_key = None
    month_rows = []

    total_done = 0
    total_skipped = 0
    total_failed = 0

    for dt in daterange(start_date, end_date):
        dstr = dt.strftime("%Y-%m-%d")

        if dstr in KNOWN_BAD_DATES:
            print(f"Skipping known bad/missing date: {dstr}")
            total_skipped += 1
            continue

        ym_key = (dt.year, dt.month)

        if current_month_key is None:
            current_month_key = ym_key

        if ym_key != current_month_key:
            append_month_chunk_to_s3(month_rows, bucket, prefix, current_month_key[0], current_month_key[1], fs)
            month_rows = []
            current_month_key = ym_key

        if ym_key not in month_cache:
            month_cache[ym_key] = list_month_tar_urls(session, dt.year, dt.month)

        month_files = month_cache[ym_key]
        if dstr not in month_files:
            print(f"No file listed for {dstr}")
            total_skipped += 1
            continue

        url = month_files[dstr]

        try:
            tar_bytes = download_tar_bytes(session, url)
            arr_mm = read_swe_mm_from_tar_bytes(tar_bytes)
            del tar_bytes

            if arr_mm is None:
                print(f"No SWE member found for {dstr}")
                total_skipped += 1
                continue

            zone_setup = zone_setup_pre if dt < SHIFT_DATE else zone_setup_post
            day_df = aggregate_day_to_huc12(arr_mm, zone_setup)
            day_df["date"] = dstr

            month_rows.append(day_df)

            del arr_mm
            total_done += 1
            print(f"Done {dstr}")

        except Exception as e:
            total_failed += 1
            print(f"Failed on {dstr}: {e}")

    if current_month_key is not None and month_rows:
        append_month_chunk_to_s3(month_rows, bucket, prefix, current_month_key[0], current_month_key[1], fs)

    print("\nPipeline summary")
    print(f"Completed: {total_done}")
    print(f"Skipped:   {total_skipped}")
    print(f"Failed:    {total_failed}")

    if combine_at_end:
        df_all = combine_monthly_parquets_from_s3(bucket, prefix, combined_key, fs)
        return df_all

    return None

In [ ]:
# Parquet written directly to S3 (monthly shards + combined)
df_all = run_snodas_huc12_pipeline(
    gdf4326=gdf,
    start_date=DATE_START,
    end_date=DATE_END,
    bucket=SNODAS_PARQUET_S3_BUCKET,
    prefix=SNODAS_S3_PREFIX,
    combined_key=SNODAS_PARQUET_S3_KEY,
    combine_at_end=True,
)

if df_all is not None:
    print(df_all.head())
    print(df_all.shape)
    print(df_all["swe_m"].min(), df_all["swe_m"].max())

In [17]:
huc_ids = df_all['huc_id'].unique()

In [18]:
huc_ids[0]

'170103020101'

## Add SNODAS SWE to model-ready CSVs

Fetch model_ready_huc{huc_id}.csv from S3, add snodas_swe column for available dates from the parquet, push back to S3.

In [9]:
def add_snodas_swe_to_model_ready(
    huc_ids: list,
    model_ready_bucket: str = MODEL_READY_BUCKET,
    parquet_bucket: str = SNODAS_PARQUET_S3_BUCKET,
    parquet_key: str = SNODAS_PARQUET_S3_KEY,
    date_col: str = "day",
) -> None:
    """
    For each model_ready_huc{huc_id}.csv in S3: add snodas_swe column from parquet
    for available dates, then upload back.
    """
    # Load SNODAS parquet from S3
    parquet_path = f"s3://{parquet_bucket}/{parquet_key}"
    swe_df = pd.read_parquet(parquet_path)
    swe_df["date"] = pd.to_datetime(swe_df["date"]).dt.normalize()  # overwrite date with normalized
    swe_df = swe_df.rename(columns={"swe_m": "snodas_swe"})[["huc_id", "date", "snodas_swe"]]

    for huc_id in huc_ids:
        huc_str = str(huc_id)
        key = f"model_ready_huc{huc_id}.csv"

        if not du.isin_s3(model_ready_bucket, key):
            print(f"Skip {key}: not in S3")
            continue

        df = du.s3_to_df(key, model_ready_bucket)
        df["date"] = pd.to_datetime(df[date_col]).dt.normalize()

        swe_sub = swe_df[swe_df["huc_id"] == huc_str][["date", "snodas_swe"]]
        df = df.merge(swe_sub, on="date", how="left")
        df = df.drop(columns=["date"])

        # Upload back (BytesIO to avoid local file, index=False)
        buf = io.BytesIO()
        df.to_csv(buf, index=False)
        buf.seek(0)
        boto3.client("s3", region_name=AWS_REGION).upload_fileobj(buf, model_ready_bucket, key)
        print(f"Updated {key}")

In [10]:
huc_ids = gdf["huc_id"].astype(str).tolist()
len(huc_ids)

1

In [11]:
# Run: add snodas_swe to model-ready CSVs (use gdf huc_ids or list S3)
add_snodas_swe_to_model_ready(
    huc_ids=huc_ids,
    model_ready_bucket=MODEL_READY_BUCKET,
    parquet_bucket=SNODAS_PARQUET_S3_BUCKET,
    parquet_key=SNODAS_PARQUET_S3_KEY,
)

Updated model_ready_huc170103020101.csv
